# Google Colab Host for EduMind AI Backend

This notebook helps you clone, configure, and host the EduMind AI backend server on Google Colab, exposing it to the public internet using **Cloudflare Tunnel (trycloudflare)** — completely free and with no sign-ups or auth tokens required.

### Step 1: Clone the Repository
We will clone the repository and switch to the correct branch (`commitee-process-and-fixes`).

In [ ]:
# 1. Clone the repository and checkout the correct branch
!git clone -b commitee-process-and-fixes https://github.com/jaynishthakar/demo-.git
%cd demo-

### Step 2: Install Dependencies
Let's install all requirements needed by the backend application.

In [ ]:
# 2. Install dependencies
!pip install -r requirements.txt
!pip install nest-asyncio

### Step 3: Configure Environment Variables
We write the `.env` configuration file containing the Ollama and LLM settings.

In [ ]:
# 3. Create or write .env file
env_content = """OLLAMA_BASE_URL=https://ken-combines-impression-leader.trycloudflare.com
NGROK_AUTHTOKEN=3FzpiDTu3BDjqq3oaOwhHZ210R8_2Gek4AzFydoBZwZA3GT6Z
OLLAMA_MODEL=qwen2.5:7b
LLM_BACKEND=ollama
HF_MODEL=Qwen/Qwen2.5-32B-Instruct
HF_TOKEN=
"""

with open(".env", "w") as f:
    f.write(env_content)
print(".env file configured successfully.")

### Step 4: Start the FastAPI Backend Server
We will run the server in the background and log its output to `backend.log`.

In [ ]:
# 4. Start the FastAPI backend server in the background
import subprocess
import time

print("Starting FastAPI backend...")
with open("backend.log", "w") as log_file:
    backend_process = subprocess.Popen(
        ["uvicorn", "backend.app:app", "--host", "0.0.0.0", "--port", "8000"],
        stdout=log_file,
        stderr=log_file
    )

time.sleep(5)  # Wait for startup

# Check if the process is still running
if backend_process.poll() is None:
    print(f"FastAPI backend started successfully in the background (PID: {backend_process.pid}).")
else:
    print("Backend failed to start. Printing backend.log:")
    with open("backend.log", "r") as log_file:
        print(log_file.read())

### Step 5: Start Cloudflare Tunnel
Let's download the Cloudflare Tunnel daemon and run it to get a public URL for our FastAPI backend.

In [ ]:
# 5. Download and start Cloudflare Tunnel (TryCloudflare)
import os
import re

# Download cloudflared CLI if not exists
if not os.path.exists("cloudflared"):
    print("Downloading Cloudflare Tunnel client (cloudflared)...")
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

print("Starting Cloudflare Tunnel...")
tunnel_process = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("Waiting for tunnel URL...")
for line in iter(tunnel_process.stdout.readline, ""):
    if "trycloudflare.com" in line:
        urls = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if urls:
            print("\n" + "="*60)
            print("YOUR BACKEND SERVER IS HOSTED PUBLICLY AT:")
            print(urls[0])
            print("="*60 + "\n")
            break